In [3]:
!pip install torch scikit-learn

In [4]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

# 检查是否有 GPU，如果有就用，没有就用 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# 1. 读取数据
# 请确保这里的文件路径和你之前可视化用到的一致
df = pd.read_csv('data/red_packet_data.csv') 

# 提取核心的三列数据：按顺序的金额 (first, second, third)
# 这一步把数据变成了 Shape 为 (100, 3) 的矩阵
raw_data = df[['first', 'second', 'third']].values.astype(np.float32)

print(f"Original Data Shape: {raw_data.shape}")
print(f"Sample raw data (first 2 rows):\n{raw_data[:2]}")

# 2. 归一化 (Min-Max Scaling to [-1, 1])
# 公式: x_norm = 2 * (x - min) / (max - min) - 1
scaler = MinMaxScaler(feature_range=(-1, 1))
normalized_data = scaler.fit_transform(raw_data)

# !!! 关键 !!!：必须保存 scaler 的参数，否则后面生成的“假数据”全是 -1 到 1 的小数，还原不回去了
data_min = scaler.data_min_
data_max = scaler.data_max_
print(f"\nScaler parameters saved.")
print(f"Data Min per column: {data_min}")
print(f"Data Max per column: {data_max}")

# 3. 构建 PyTorch Dataset 和 DataLoader
# 将 numpy 数组转换为 PyTorch Tensor
tensor_data = torch.FloatTensor(normalized_data).to(device)

# 创建 Dataset
dataset = TensorDataset(tensor_data)

# 创建 DataLoader
# batch_size=32 对于 100 条数据来说稍微有点大，但也能跑。
# 如果想要模型多更新几次参数，可以改小一点，比如 16。
BATCH_SIZE = 16 
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# 4. 验证一下 DataLoader 输出
# 取出一个 batch 看看长什么样
sample_batch = next(iter(dataloader))[0]
print(f"\nDataLoader ready!")
print(f"Batch Shape: {sample_batch.shape} (Should be [{BATCH_SIZE}, 3])")
print(f"Sample normalized batch (approx range [-1, 1]):\n{sample_batch[:2].cpu().numpy()}")